# Audit fixed-time-lag motion changes and sampling support

The twenty-epoch HR reference does not improve the primary score. Keep ten past HR epochs and both HR channels.
Next question: can a declared fixed-time-lag axis-change diagnostic address the cadence ambiguity found in notebook 28?
Compare the current full-HR reference and the completed twenty-epoch candidate on the same development targets.
This notebook performs descriptive sensor analysis only. It cannot train a model or integrate a feature into training.
The raw audit reads checksum-verified motion for the 26 development participants. The three final-test participants remain excluded.
At each epoch, inspect one-second grid times from +1 through +29 seconds. Use only earlier-or-equal observations within that epoch.
Require reading age at most 0.2 seconds. Compare adjacent supported grid vectors and retain support and actual sample-lag metadata.
These are source-unit axis-vector changes, not jerk or physical displacement. The grid can miss faster activity between ticks.
Read [the protocol](../docs/motion-grid-audit.md). Keep RUN_AUDIT false when the cached snapshot exists.


## Locate the checkout and declare execution gates

**What:** Find the repository, configure notebook logging, and declare setup and audit switches.

**Why:** Report-only execution must reuse verified cached evidence. The raw-data audit needs explicit opt-in and never trains a model.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import JSON, FileLink, display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("motion_grid_audit_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/motion-grid-audit.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "analysis/motion-grid-audit"  # Select this PR branch before merge
RUN_SETUP = False
RUN_AUDIT = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s raw-data audit=%s; no model optimization is available", RUN_SETUP, RUN_AUDIT)


## Optional environment setup

**What:** Optionally clone the working branch and install Astral uv with locked analysis and notebook dependencies.

**Why:** A new machine needs the project modules and plotting tools. The existing local environment can skip this cell.
The audit also needs the prepared development features, completed experiment artifacts, and checksum-verified development motion files.
This environment does not need the optional neural-network dependency group.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group baseline --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the development-only audit plan

**What:** Read the configured study metadata and inspect participant membership, recipes, and raw-file scope.

**Why:** Scope checks must precede observation access. The plan must exclude the reserved test participants and expose no training action.
The existing eligibility rules, model inputs, saved predictions, and model parameters remain unchanged.


In [ ]:
import pandas as pd

from sleep_classifier.causal_feature_audit import CausalFeatureAudit

experiment = CausalFeatureAudit(ROOT / "configs/motion-grid-audit.yaml")
plan = experiment.plan()
development = set(plan["development_participants"])
reserved_test = set(plan["reserved_test_participants"])
if len(development) != 26 or len(reserved_test) != 3 or development & reserved_test:
    raise ValueError("The audit must use the frozen development cohort and reserve the test set")
if (plan["model_training_performed"] is not False
        or plan["test_observations_read"] is not False
        or plan["candidate_features_integrated"] is not False):
    raise ValueError("The audit must remain read-only with respect to models and final-test data")
display(JSON(plan, expanded=False))
logger.info("Audit plan verified: %d development participants, %d reserved test participants",
            len(development), len(reserved_test))

logger.info("Audit identity: config=%s output=%s motion_grid=%s",
            experiment.settings.config_path, experiment.output_dir,
            experiment.settings.values["motion_grid"])


## Understand why adjacent change depends on cadence

**What:** Sample the same linear vector trajectory at two ideal cadences. Compare adjacent and grid-based means through the shared implementations.

**Why:** A change per observation shrinks when observations are closer together. A fixed time lag makes that comparison explicit on this constructed trajectory.


In [ ]:
import numpy as np

from sleep_classifier.audit_features import CausalAuditFeatures
from sleep_classifier.motion_grid import FixedLagMotionFeatures

grid = FixedLagMotionFeatures(step_seconds=1.0, max_sample_age_seconds=0.2)
epoch = pd.DataFrame({"time": [0.0]})
example_rows = []
for cadence in (0.02, 0.1):
    times = np.round(np.arange(0, 30, cadence), 8)
    motion = pd.DataFrame({"time": times, "x": times, "y": 0.0, "z": 1.0})
    adjacent = CausalAuditFeatures().motion_summaries(motion, epoch).iloc[0]
    fixed = grid.motion_summaries(motion, epoch).iloc[0]
    example_rows.append({"sample_gap_seconds": cadence,
                         "adjacent_change": adjacent.motion_vector_change_mean,
                         "one_second_change": fixed.motion_grid_change_mean,
                         "supported_grid_pairs": fixed.motion_grid_pair_count})
display(pd.DataFrame(example_rows))
logger.info("Ideal trajectory only: sensor filtering and cadence independence remain unproven")


## Understand added information and aliasing limits

**What:** Compare constant-magnitude axis patterns that differ between or at the one-second ticks.

**Why:** Magnitude summaries discard direction. Fixed-lag changes can expose some direction changes, but miss activity returning to the same vector between ticks.


In [ ]:
times = np.round(np.arange(0, 30, 0.02), 8)
patterns = {
    "constant_vector": np.zeros(len(times), dtype=bool),
    "switch_at_each_second": np.floor(times).astype(int) % 2 == 1,
    "return_before_next_tick": times % 1 >= 0.5,
}
pattern_rows = []
for name, switch in patterns.items():
    motion = pd.DataFrame({"time": times, "x": (~switch).astype(float),
                           "y": switch.astype(float), "z": 0.0})
    fixed = grid.motion_summaries(motion, epoch).iloc[0]
    magnitudes = np.linalg.norm(motion[["x", "y", "z"]].to_numpy(), axis=1)
    pattern_rows.append({"pattern": name, "magnitude_std": float(magnitudes.std()),
                         "one_second_change": fixed.motion_grid_change_mean})
display(pd.DataFrame(pattern_rows))
logger.info("A zero fixed-lag value does not prove absence of faster motion between grid ticks")


## Create one immutable development-only snapshot

**What:** Optionally read permitted raw motion once and create a new audit snapshot.

**Why:** Raw cadence and fixed-lag support require sensor observations. The cached report below needs no raw reread.
The command uses the validated audit config. Existing snapshots are verified and reused. Invalid or partial snapshots stop execution without overwriting evidence.


In [ ]:
if RUN_AUDIT:
    audit_config = experiment.settings.config_path
    logger.info("Audit intent: config=%s output=%s", audit_config, experiment.output_dir)
    if experiment.output_dir.exists():
        experiment.load_report()
        logger.info("Verified existing snapshot; skipping raw audit: %s", experiment.output_dir)
    else:
        %cd "{ROOT}"
        !uv run --locked --group baseline --group notebook python -u \
            -m sleep_classifier.causal_feature_audit --config "{audit_config}" --run
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Motion grid audit failed; inspect retained output before retrying")
else:
    logger.info("Raw audit skipped; load the verified cached snapshot below")


## Load and verify the cached audit

**What:** Load the cached target table and summary through the audit reader's output-hash checks.

**Why:** A dashboard must describe a consistent snapshot rather than silently combine changed files.
The reader verifies registered output files. The participant check below also guards against an unexpected test-set row.
Report-only execution does not rerun raw-file processing when the cache is absent.


In [ ]:
rows = None
summary = None
cached_paths = [experiment.output_dir / "manifest.json", experiment.output_dir / "summary.json"]
if any(path.exists() for path in cached_paths) and not all(path.exists() for path in cached_paths):
    raise RuntimeError("The cached audit is partial; inspect it before creating another snapshot")
if all(path.exists() for path in cached_paths):
    rows, summary = experiment.load_report()
    observed = set(rows["participant_id"].astype(str))
    if not observed <= development or observed & reserved_test:
        raise ValueError("The cached audit contains a participant outside the development scope")
    overview_keys = (
        "created_utc", "participants", "targets", "scope", "test_observations_read",
        "model_training_performed", "candidate_features_integrated",
    )
    display(JSON({key: summary[key] for key in overview_keys}, expanded=True))
    logger.info("Provenance records %d saved-artifact hashes and %d development motion reads",
                len(summary["source_artifact_hashes"]), len(summary["raw_files_read"]))
    logger.info("Verified cached audit: %d targets from %d development participants",
                len(rows), len(observed))
else:
    logger.info("No cached audit exists; enable RUN_AUDIT only when ready for development analysis")


## Inspect reference errors without ranking participants

**What:** Show the related-recipe persistent Deep/REM errors and current reference error counts by participant.

**Why:** Repeated model agreement is correlated evidence. Counts reflect support as well as difficulty.
The reference is the working ten-epoch full-HR model. The comparison is the completed twenty-epoch candidate.


In [ ]:
if rows is not None:
    display(pd.DataFrame(summary["persistent_counts"]))
    participant_counts = rows.groupby("participant_id", sort=True).agg(
        eligible_epochs=("time", "size"),
        persistent_to_light=("persistent_to_light", "sum"),
        reference_accuracy=("reference_correct", "mean"),
    ).sort_values("persistent_to_light", ascending=False)
    display(participant_counts)
else:
    logger.info("Persistent-error tables are pending the first audit snapshot")


## Inspect support, cadence, and error contrasts

**What:** Show label-free support sensitivity, participant cadence/support, stage availability, and within-participant reference-error summaries.

**Why:** A large diagnostic value is not useful evidence without adequate support. Sensor cadence and missingness can differ by person or stage.
Support fractions 0.5, 0.8, and 0.95 are descriptive sensitivity levels, not chosen model-eligibility thresholds.
No target is removed. Label-conditioned contrasts remain post-hoc diagnostics and never become model inputs.


In [ ]:
if summary is not None:
    display(JSON(summary["motion_grid"], expanded=True))
    for key in ("candidate_availability", "support_sensitivity", "participant_cadence_support",
                "stage_feature_availability", "reference_error_summary"):
        logger.info("Descriptive table: %s", key)
        display(pd.DataFrame(summary[key]))
else:
    logger.info("Sampling and support tables are pending the first immutable snapshot")


## Compare feature distributions at participant level

**What:** Plot one median per participant and true stage for the original and proposed diagnostics.

**Why:** Many neighboring epochs are correlated. Per-person summaries reduce the temptation to treat epochs as independent people.
Visual separation does not establish a classification gain. Check support and cadence beside the feature values.


In [ ]:
import matplotlib.pyplot as plt

from sleep_classifier.audit_visualization import AuditVisualization

if rows is not None:
    visualization = AuditVisualization(rows, summary)
    for figure in (visualization.plot_distributions(), visualization.plot_persistent_errors()):
        display(figure)
        plt.close(figure)
else:
    logger.info("Audit figures are pending the first verified snapshot")


## Open the portable sampling-aware viewer

**What:** Link the verified offline dashboard with grid-change, support, age, and actual-lag views.

**Why:** Participant timelines reveal gaps and failure modes that aggregate distributions hide. The viewer makes no network requests.


In [ ]:
if rows is not None:
    dashboard = (experiment.output_dir / "dashboard.html").resolve()
    if not dashboard.is_file():
        raise FileNotFoundError("The verified audit snapshot lacks its dashboard")
    display(FileLink(str(dashboard), result_html_prefix="Portable dashboard: "))
else:
    logger.info("The dashboard link is pending the first audit snapshot")


## Record the decision boundary

**What:** Summarize what this audit can establish before proposing one feature ablation.

**Why:** Adaptive development analysis must not be presented as independent final-test evidence or an immediate model improvement.
Inspect persistent errors, candidate support, participant variation, and the train-only descriptive summaries together.
Choose one feature family only after stating a causal, testable hypothesis and its handling of missing history or sample gaps.
Keep the original Adam/P3 recipe fixed for that later comparison. Keep all final-test observations untouched.
No new feature is integrated into the training feature schema by this notebook.


In [ ]:
if summary is not None:
    logger.info("Audit complete: no model optimization, feature promotion, or final-test reads")
    logger.info("Review cadence, support, aliasing, and Light contrasts before an ablation")
else:
    logger.info("Decision pending: run the development-only audit and review its support evidence")


## Export the executed notebook as HTML

**What:** Execute the source notebook from a fresh kernel and export its verified cached report as standalone HTML.

**Why:** Keep an inspectable report while source notebooks remain output-free in Git.
Automatic export skips setup, the raw-data audit, and this recursive export cell.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group baseline --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/35_motion_grid_audit.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Causal feature audit notebook export failed")


## Observed development audit
All 22,708 targets have a supported grid-change estimate. At least 80% point/pair support holds for every target.
The 95% check retains 22,685 targets descriptively; no target is removed from the audit.
Stage distributions overlap. The audit validates feasibility, not classification benefit.
See [the measured findings](../docs/experiments/motion-grid-audit-2026-11-19/README.md) for limits and the next decision.